# Module 06 — RAG From First Principles

**Predict → Build → Try → Break → Measure → Improve.** Build a framework-free retrieval pipeline and separate retrieval quality from generation quality.

## Architecture

Documents → chunks + metadata → deterministic embedding → exact vector search → evidence → grounded answer/abstention.

Authorization and tenant filtering happen before ranking/truncation; the model never becomes the authorization layer.

In [ ]:
import sys
sys.path.insert(0, '..')
from app.models import Chunk
from app.index import VectorIndex
from app.embeddings import cosine_similarity


## BUILD — deterministic RAG baseline

Create a tiny multi-tenant corpus, index coherent chunks, then retrieve with a metadata filter.

In [ ]:
index = VectorIndex()
chunks = [
    Chunk('c1','d1','Vacation policy: employees receive 20 days of annual leave.', {'tenant_id':'a','section':'leave'}),
    Chunk('c2','d2','Security policy: production credentials require approval.', {'tenant_id':'a','section':'security'}),
    Chunk('c3','d3','Vacation policy: employees receive 25 days of annual leave.', {'tenant_id':'b','section':'leave'}),
]
for chunk in chunks: index.add(chunk)
results = index.search('How many annual leave days?', top_k=3, metadata_filter={'tenant_id':'a'})
[(r.chunk.chunk_id, round(r.score, 3), r.chunk.text) for r in results]


## TRY — inspect evidence

Predict which chunk should rank first. Explain why the tenant filter is part of correctness, not an optimization.

In [ ]:
assert results and all(r.chunk.metadata['tenant_id'] == 'a' for r in results)
print('Top evidence:', results[0].chunk.chunk_id, results[0].chunk.text)


## BREAK — retrieval failure experiments

1. Remove the tenant filter and demonstrate that cross-tenant evidence can become eligible.
2. Change the query to an unrelated question and decide whether the system should abstain.
3. Replace coherent chunks with giant noisy text and predict the retrieval effect.
4. Treat the model's answer as truth and then contrast it with evidence IDs.

In [ ]:
unsafe = index.search('annual leave days', top_k=3)
assert any(r.chunk.metadata['tenant_id'] == 'b' for r in unsafe)
print('BREAK reproduced: unfiltered retrieval admits another tenant.')


## MEASURE — retrieval quality

Use labeled relevant chunk IDs to compute Recall@K and MRR. Also record latency, token budget and cost/task in a production benchmark.


In [ ]:
relevant = {'q1': {'c1'}}
ranked = [r.chunk.chunk_id for r in results]
recall_at_3 = int(bool(set(ranked[:3]) & relevant['q1']))
first_rank = next((i+1 for i,c in enumerate(ranked) if c in relevant['q1']), None)
mrr = 1/first_rank if first_rank else 0.0
print({'Recall@3': recall_at_3, 'MRR': mrr})


## EXERCISE — defend the design

Compare vector retrieval with a deterministic database lookup and long-context prompting. State the conditions under which RAG is the wrong architecture. Then add citation validation and an abstention threshold.

**Mastery gate:** diagnose whether a wrong answer came from ingestion, retrieval, context assembly or generation, using evidence rather than guesswork.